In [13]:

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv, find_dotenv


import warnings
warnings.filterwarnings('ignore')


from dotenv import load_dotenv, find_dotenv
import os

from langchain_core.caches import InMemoryCache
# from langchain_openai import OpenAI
from openai import OpenAI
from langchain_openai import ChatOpenAI
import langchain

from langchain_core.messages import (
    AIMessage, 
    HumanMessage, 
    SystemMessage
)
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder
)

from langchain_community.chat_message_histories.in_memory import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_community.document_loaders import *
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai  import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

from typing import Annotated, TypedDict

from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END

from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import ToolNode, tools_condition



In [14]:
load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
tavily = os.getenv("TAVILY_API_KEY")
llm_name = "gpt-4o-mini"


In [15]:
chat_model = OpenAI(api_key=OPENAI_API_KEY)


In [16]:
llm_name = "gpt-4o-mini"

client = OpenAI(api_key=OPENAI_API_KEY)
model = ChatOpenAI(api_key=OPENAI_API_KEY, model=llm_name)

In [17]:

class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

In [18]:
# create tools
tool = TavilySearchResults(max_results=2)
tools = [tool]
# rest = tool.invoke("What is the capital of France?")
# print(rest)

model_with_tools = model.bind_tools(tools)

In [19]:
def bot(state: State):
    # print(state.items())
    print(state["messages"])
    return {"messages": [model_with_tools.invoke(state["messages"])]}

In [ ]:

graph_builder = StateGraph(State)

tool_node = ToolNode(tools=[tool])
graph_builder.add_node("tools", tool_node) 


# The `tools_condition` function returns "tools" if the chatbot asks to use a tool, and "__end__" if
# it is fine directly responding. This conditional routing defines the main agent loop.
graph_builder.add_conditional_edges(
    "bot",
    tools_condition,
)

# The first argument is the unique node name
# The second argument is the function or object that will be called whenever
# the node is used.
graph_builder.add_node("bot", bot)


# STEP 3: Add an entry point to the graph
graph_builder.set_entry_point("bot")